### Customer Distribution Analysis - Gold Layer

In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.customers;

In [0]:
%sql
SELECT customer_state AS state,
  COUNT(DISTINCT customer_city) AS distinct_cities
FROM course_training_catalog.silver_olist.customers
GROUP BY state
ORDER BY distinct_cities DESC
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE course_training_catalog.gold_olist.Customer_Top_10_States_by_City_diversity AS
SELECT customer_state AS state,
  COUNT(DISTINCT customer_city) AS distinct_cities
FROM course_training_catalog.silver_olist.customers
GROUP BY state
ORDER BY distinct_cities DESC
LIMIT 10;

In [0]:
%sql
SELECT * FROM course_training_catalog.gold_olist.customer_top_5_states_by_city_diversity

Databricks visualization. Run in Databricks to view.

- Goal of this particular exercise?
- For each state we're going to identify how many different cities customers come from.
- Average number of customers per city, density, the city with the highest number of customers and its count.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customer_diversity_with_top_city AS
WITH city_counts AS(
  SELECT
    customer_state AS state,
    customer_city AS city,
    COUNT(*) AS customer_count
  FROM 
    course_training_catalog.silver_olist.customers
  GROUP BY
    state,
    city
),
ranked AS(
  SELECT
    state,
    city,
    customer_count,
    ROW_NUMBER() OVER(PARTITION BY state ORDER BY customer_count DESC) AS rn
  FROM
    city_counts
),
state_summary AS(
  SELECT
    state,
    COUNT(DISTINCT city) AS distinct_cities,
    SUM(customer_count) AS total_customers,
    ROUND(AVG(customer_count), 2) AS avg_customers_per_city
  FROM city_counts
  GROUP BY state
)
SELECT
  s.state,
  s.distinct_cities,
  s.total_customers,
  s.avg_customers_per_city,
  r.city AS top_city,
  r.customer_count AS top_city_customers
FROM state_summary s
JOIN ranked r
  ON s.state = r.state AND r.rn == 1
ORDER BY s.total_customers DESC;


SELECT * FROM customer_diversity_with_top_city;


In [0]:
%sql
CREATE OR REPLACE TABLE course_training_catalog.gold_olist.customer_distribution_by_state AS
SELECT * FROM customer_diversity_with_top_city;

In [0]:
%sql
SELECT * FROM course_training_catalog.gold_olist.customer_distribution_by_state;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### Seller Metrics and Pareto Visualization in Databricks

In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.sellers;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW seller_clean AS
SELECT
  INITCAP(seller_city) AS city_display,
  UPPER(seller_state) AS state_code
FROM
  course_training_catalog.silver_olist.sellers
WHERE seller_id IS NOT NULL; 
-- note that we have already cleared these in the silver table, uppercasing, trimming, type conversions, deduplication. here we are doing it again for two reasons, to make sure the display looks good and in a visually appealing format.


In [0]:
%sql
SELECT * FROM seller_clean;

In [0]:
%sql
CREATE OR REPLACE TABLE course_training_catalog.gold_olist.seller_city_stats AS
WITH city_counts AS(
  SELECT
    city_display,
    state_code,
    COUNT(*) AS seller_count
    FROM seller_clean
    GROUP BY
      city_display,
      state_code
),
ranked AS(
  SELECT
    city_display,
    state_code,
    seller_count,
    DENSE_RANK() OVER(ORDER BY seller_count DESC, city_display) AS rnk,
    SUM(seller_count) OVER() AS total_all,
    SUM(seller_count) OVEr(ORDER BY seller_count DESC, city_display ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_sellers
  FROM
    city_counts
)
SELECT
  city_display,
  state_code,
  seller_count,
  rnk,
  ROUND(cum_sellers / total_all, 3) AS cum_share
  FROM ranked;

In [0]:
%sql
SELECT * FROM course_training_catalog.gold_olist.seller_city_stats;

In [0]:
%sql
SELECT * FROM course_training_catalog.gold_olist.seller_city_stats
WHERE rnk <= 20
ORDER BY rnk;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.products;

In [0]:
import pyspark.sql.functions as F

df_products = spark.table("course_training_catalog.silver_olist.products")

df_summary = (
    df_products
    .groupBy("product_category_name")
    .agg(
        F.round(F.avg("product_weight_g"), 2).alias("avg_weight_g"),
        F.round(F.avg("product_length_cm"), 2).alias("avg_length_cm"),
        F.round(F.avg("product_height_cm"), 2).alias("avg_height_cm"),
        F.round(F.avg("product_width_cm"), 2).alias("avg_width_cm"),
        F.round(F.avg("product_volume_cm3"), 2).alias("avg_volume_cm3"),
        F.round(F.avg(F.col("product_weight_g") / F.col("product_volume_cm3")), 3).alias("avg_density")
    )
    .filter(F.col("product_category_name").isNotNull())
    .orderBy(F.desc("avg_weight_g"))
)

In [0]:
display(df_summary)

In [0]:
df_summary.write.mode("overwrite").format("delta") \
    .saveAsTable("course_training_catalog.gold_olist.product_category_summary")

In [0]:
%sql
SELECT * FROM course_training_catalog.gold_olist.product_category_summary LIMIT 10;

In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.category_lookup;

In [0]:
df = (
    spark.table("course_training_catalog.gold_olist.product_category_summary").alias("p")
    .join(spark.table("course_training_catalog.silver_olist.category_lookup").alias("l"),
          F.col("p.product_category_name") == F.col("l.category_pt"), "left")
    .selectExpr(
        "p.product_category_name AS category_pt",
        "COALESCE(l.category_en, p.product_category_name) AS category_en",
        "initcap(regexp_replace(COALESCE(l.category_en, p.product_category_name), ' ', '_')) AS category_en_display",
        "avg_weight_g",
        "avg_length_cm",
        "avg_height_cm",
        "avg_width_cm",
        "avg_volume_cm3",
        "avg_density"
    )
)

In [0]:
df.write.mode("overwrite").format("delta") \
    .saveAsTable("course_training_catalog.gold_olist.product_category_summary_en")

In [0]:
%sql
SELECT * FROM course_training_catalog.gold_olist.product_category_summary_en;

In [0]:
%sql
SELECT * FROM course_training_catalog.gold_olist.product_category_summary_en
ORDER BY avg_weight_g DESC
LIMIT 15;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT * FROM course_training_catalog.gold_olist.product_category_summary_en
ORDER BY avg_weight_g DESC
LIMIT 25;

Databricks visualization. Run in Databricks to view.